# Early-Pandemic COVID-19 Surveillance Sample

Downloads a reproducible 10,000-record computational subsample from the CDC COVID-19 case surveillance API for March 15 through December 10, 2020.

**Sampling design:** randomly select 20 calendar dates (seed 42), then retrieve a reproducibly selected block of up to 500 records per date. This is a computational subsample, not a simple random sample of all eligible surveillance records.

The notebook enforces a fixed 12-column schema and validates each chunk **before** writing it to disk.

In [1]:
import requests
import random
import pandas as pd
import time
import os

from datetime import date, timedelta
from requests.exceptions import RequestException

## Study and sampling settings

In [2]:
API_URL = "https://data.cdc.gov/resource/vbim-akqf.json"

START_DATE = date(2020, 3, 15)
END_DATE = date(2020, 12, 10)

SEED = 42
random.seed(SEED)

N_DATES = 20
CASES_PER_DATE = 500

OUTPUT_FILE = "covid_early_pandemic_sample_10k.csv"
PROGRESS_FILE = "completed_dates_10k.txt"

MAX_RETRIES = 5

COLUMNS = [
    "cdc_case_earliest_dt",
    "cdc_report_dt",
    "onset_dt",
    "current_status",
    "sex",
    "age_group",
    "race_ethnicity_combined",
    "hosp_yn",
    "icu_yn",
    "death_yn",
    "medcond_yn",
    "pos_spec_dt"
]

## Select study dates

In [3]:
study_dates = []
current_date = START_DATE

while current_date <= END_DATE:
    study_dates.append(current_date)
    current_date += timedelta(days=1)

print(f"Study period contains {len(study_dates)} days")

selected_dates = sorted(
    random.sample(study_dates, N_DATES)
)

print(f"Selected {len(selected_dates)} dates")

for selected_date in selected_dates:
    print(selected_date)

Study period contains 271 days
Selected 20 dates
2020-03-27
2020-03-28
2020-03-30
2020-03-31
2020-04-28
2020-05-01
2020-05-06
2020-05-11
2020-05-25
2020-06-24
2020-07-04
2020-07-05
2020-07-07
2020-07-12
2020-07-18
2020-08-02
2020-10-15
2020-10-17
2020-10-30
2020-11-28


## Request helper

In [4]:
def get_with_retry(session, params, timeout):
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = session.get(
                API_URL,
                params=params,
                timeout=timeout
            )
            response.raise_for_status()
            return response

        except RequestException as error:
            print(
                f"Request failed "
                f"(attempt {attempt}/{MAX_RETRIES})"
            )
            print(error)

            if attempt == MAX_RETRIES:
                raise

            wait_time = attempt * 5
            print(f"Retrying in {wait_time} seconds...")
            time.sleep(wait_time)

## Chunk validation

In [5]:
VALID_VALUES = {
    "current_status": {
        "Laboratory-confirmed case",
        "Probable Case"
    },
    "sex": {
        "Male",
        "Female",
        "Unknown",
        "Other",
        "Missing"
    },
    "hosp_yn": {
        "Yes",
        "No",
        "Unknown",
        "Missing"
    },
    "icu_yn": {
        "Yes",
        "No",
        "Unknown",
        "Missing"
    },
    "death_yn": {
        "Yes",
        "No",
        "Unknown",
        "Missing"
    },
    "medcond_yn": {
        "Yes",
        "No",
        "Unknown",
        "Missing"
    }
}


def validate_chunk(chunk_df):
    if list(chunk_df.columns) != COLUMNS:
        raise ValueError(
            "Chunk columns do not match expected schema."
        )

    for column, valid_values in VALID_VALUES.items():
        observed_values = set(
            chunk_df[column]
            .dropna()
            .unique()
        )

        invalid_values = observed_values - valid_values

        if invalid_values:
            raise ValueError(
                f"Invalid values found in {column}: "
                f"{invalid_values}"
            )

## Download sample

This uses new 10k-specific output/progress filenames, so it does not append to the corrupted 100k file. If interrupted, rerunning this cell skips dates already saved successfully.

In [6]:
completed_dates = set()

if os.path.exists(PROGRESS_FILE):
    with open(PROGRESS_FILE, "r") as file:
        completed_dates = {
            line.strip()
            for line in file
            if line.strip()
        }

print(f"Previously completed dates: {len(completed_dates)}")

session = requests.Session()

for date_number, selected_date in enumerate(
    selected_dates,
    start=1
):
    date_string = selected_date.isoformat()

    if date_string in completed_dates:
        print(
            f"[{date_number}/{N_DATES}] "
            f"{date_string} already completed"
        )
        continue

    next_date = selected_date + timedelta(days=1)

    start_timestamp = f"{date_string}T00:00:00"
    end_timestamp = f"{next_date.isoformat()}T00:00:00"

    where_clause = f"""
        cdc_case_earliest_dt >= '{start_timestamp}'
        AND cdc_case_earliest_dt < '{end_timestamp}'
    """

    print(f"[{date_number}/{N_DATES}] {date_string}")

    # Count eligible cases for this date
    count_params = {
        "$select": "count(*) as count",
        "$where": where_clause
    }

    response = get_with_retry(
        session=session,
        params=count_params,
        timeout=180
    )

    daily_count = int(response.json()[0]["count"])

    print(f"          eligible cases: {daily_count:,}")

    # Choose a reproducible starting point within the date
    if daily_count <= CASES_PER_DATE:
        offset = 0
        limit = daily_count
    else:
        max_offset = daily_count - CASES_PER_DATE

        # Create reproducible RNG unique to this date
        date_seed = SEED + selected_date.toordinal()
        date_rng = random.Random(date_seed)

        offset = date_rng.randint(
            0,
            max_offset
        )
        limit = CASES_PER_DATE

    # Retrieve sampled records
    params = {
        "$where": where_clause,
        "$order": ":id ASC",
        "$limit": limit,
        "$offset": offset
    }

    response = get_with_retry(
        session=session,
        params=params,
        timeout=300
    )

    rows = response.json()

    print(
        f"          downloaded {len(rows):,} cases "
        f"(offset {offset:,})"
    )

    # Convert API output to dataframe
    chunk_df = pd.DataFrame(rows)

    # Enforce consistent column schema before saving
    chunk_df = chunk_df.reindex(columns=COLUMNS)

    # Validate before writing anything to disk
    validate_chunk(chunk_df)

    # Check if output file already exists
    file_exists = os.path.exists(OUTPUT_FILE)

    # Save downloaded records immediately
    chunk_df.to_csv(
        OUTPUT_FILE,
        mode="a",
        header=not file_exists,
        index=False
    )

    # Mark date complete only after successful save
    with open(PROGRESS_FILE, "a") as file:
        file.write(date_string + "\n")

    completed_dates.add(date_string)

    print("          saved to disk")

    time.sleep(0.25)

print("\nDownload finished.")

Previously completed dates: 0
[1/20] 2020-03-27
          eligible cases: 25,356
          downloaded 500 cases (offset 611)
          saved to disk
[2/20] 2020-03-28
          eligible cases: 20,437
          downloaded 500 cases (offset 3,034)
          saved to disk
[3/20] 2020-03-30
          eligible cases: 30,037
          downloaded 500 cases (offset 27,381)
          saved to disk
[4/20] 2020-03-31
          eligible cases: 26,322
          downloaded 500 cases (offset 2,834)
          saved to disk
[5/20] 2020-04-28
          eligible cases: 31,154
          downloaded 500 cases (offset 28,051)
          saved to disk
[6/20] 2020-05-01
          eligible cases: 30,717
          downloaded 500 cases (offset 17,381)
          saved to disk
[7/20] 2020-05-06
          eligible cases: 27,197
          downloaded 500 cases (offset 16,864)
          saved to disk
[8/20] 2020-05-11
          eligible cases: 28,448
          downloaded 500 cases (offset 12,661)
          saved to disk

## Final structural validation

In [7]:
df = pd.read_csv(OUTPUT_FILE)

expected_max_rows = N_DATES * CASES_PER_DATE

print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")
print(f"Maximum expected rows: {expected_max_rows:,}")

print("\nColumn order:")
for column in df.columns:
    print(column)

assert list(df.columns) == COLUMNS, (
    "Final CSV columns do not match the expected schema."
)

assert len(df) <= expected_max_rows, (
    "Final CSV contains more rows than the sampling design permits."
)

print("\nStructural validation passed.")

Rows: 10,000
Columns: 12
Maximum expected rows: 10,000

Column order:
cdc_case_earliest_dt
cdc_report_dt
onset_dt
current_status
sex
age_group
race_ethnicity_combined
hosp_yn
icu_yn
death_yn
medcond_yn
pos_spec_dt

Structural validation passed.


## Categorical QC

In [8]:
categorical_columns = [
    "current_status",
    "sex",
    "age_group",
    "race_ethnicity_combined",
    "hosp_yn",
    "icu_yn",
    "death_yn",
    "medcond_yn"
]

for column in categorical_columns:
    print(f"\n{column}:")
    print(df[column].value_counts(dropna=False))


current_status:
current_status
Laboratory-confirmed case    9465
Probable Case                 535
Name: count, dtype: int64

sex:
sex
Male       5461
Female     4516
Missing      14
Unknown       9
Name: count, dtype: int64

age_group:
age_group
30 - 39 Years    2467
50 - 59 Years    1428
60 - 69 Years    1281
20 - 29 Years    1245
40 - 49 Years    1059
0 - 9 Years      1016
70 - 79 Years     628
10 - 19 Years     472
80+ Years         396
Missing             8
Name: count, dtype: int64

race_ethnicity_combined:
race_ethnicity_combined
White, Non-Hispanic                                     3454
Unknown                                                 3128
Hispanic/Latino                                         1972
Black, Non-Hispanic                                      765
Asian, Non-Hispanic                                      369
Multiple/Other, Non-Hispanic                             185
Missing                                                   81
Native Hawaiian/Other Pacific

## Date QC

In [9]:
case_dates = pd.to_datetime(
    df["cdc_case_earliest_dt"],
    errors="coerce"
)

print(
    "Observed date range:",
    case_dates.min(),
    "through",
    case_dates.max()
)

assert case_dates.notna().all(), (
    "At least one case date could not be parsed."
)

assert (case_dates.dt.date >= START_DATE).all(), (
    "A record occurs before the study period."
)

assert (case_dates.dt.date <= END_DATE).all(), (
    "A record occurs after the study period."
)

print("Date validation passed.")

Observed date range: 2020-03-27 00:00:00 through 2020-11-28 00:00:00
Date validation passed.


# Instersection of Bands of Interest

In [10]:
# Create completeness indicators
df["medcond_complete"] = (
    df["medcond_yn"].isin(["Yes", "No"])
)

df["hosp_complete"] = (
    df["hosp_yn"].isin(["Yes", "No"])
)

df["analysis_complete"] = (
    df["medcond_complete"]
    & df["hosp_complete"]
)

print("Exposure completeness:")
print(
    df["medcond_complete"]
    .value_counts()
)

print(
    df["medcond_complete"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nOutcome completeness:")
print(
    df["hosp_complete"]
    .value_counts()
)

print(
    df["hosp_complete"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nExposure + outcome completeness:")
print(
    df["analysis_complete"]
    .value_counts()
)

print(
    df["analysis_complete"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Exposure completeness:
medcond_complete
False    8184
True     1816
Name: count, dtype: int64
medcond_complete
False    81.84
True     18.16
Name: proportion, dtype: float64

Outcome completeness:
hosp_complete
True     5743
False    4257
Name: count, dtype: int64
hosp_complete
True     57.43
False    42.57
Name: proportion, dtype: float64

Exposure + outcome completeness:
analysis_complete
False    8536
True     1464
Name: count, dtype: int64
analysis_complete
False    85.36
True     14.64
Name: proportion, dtype: float64


$
P(Y\text{ observed}) = 57.43\% \\
P(E\text{ observed}) = 18.16\% \\
P(E\text{ and }Y\text{ observed}) = 14.64\%
$

## Specific Band Analysis

### Age

In [12]:
# Analysis completeness by age group
age_completeness = pd.crosstab(
    df["age_group"],
    df["analysis_complete"],
    normalize="index"
).mul(100).round(2)

print(age_completeness)

analysis_complete   False  True 
age_group                       
0 - 9 Years         85.33  14.67
10 - 19 Years       81.36  18.64
20 - 29 Years       80.00  20.00
30 - 39 Years       89.38  10.62
40 - 49 Years       90.75   9.25
50 - 59 Years       86.27  13.73
60 - 69 Years       78.53  21.47
70 - 79 Years       90.76   9.24
80+ Years           77.53  22.47
Missing            100.00   0.00


### Sex

In [13]:
# Analysis completeness by sex
sex_completeness = pd.crosstab(
    df["sex"],
    df["analysis_complete"],
    normalize="index"
).mul(100).round(2)

print(sex_completeness)

analysis_complete  False  True 
sex                            
Female             84.61  15.39
Male               85.95  14.05
Missing            92.86   7.14
Unknown            88.89  11.11


### Race/Ethnicity

In [14]:
# Analysis completeness by race and ethnicity
race_completeness = pd.crosstab(
    df["race_ethnicity_combined"],
    df["analysis_complete"],
    normalize="index"
).mul(100).round(2)

print(race_completeness)

analysis_complete                                    False  True 
race_ethnicity_combined                                          
American Indian/Alaska Native, Non-Hispanic         100.00   0.00
Asian, Non-Hispanic                                  77.78  22.22
Black, Non-Hispanic                                  84.71  15.29
Hispanic/Latino                                      89.30  10.70
Missing                                             100.00   0.00
Multiple/Other, Non-Hispanic                         87.57  12.43
Native Hawaiian/Other Pacific Islander, Non-His...   55.26  44.74
Unknown                                              94.82   5.18
White, Non-Hispanic                                  75.33  24.67


### Samples by Months

In [15]:
# Create case month
df["case_month"] = pd.to_datetime(
    df["cdc_case_earliest_dt"]
).dt.to_period("M")

# Analysis completeness by month
month_completeness = pd.crosstab(
    df["case_month"],
    df["analysis_complete"],
    normalize="index"
).mul(100).round(2)

print(month_completeness)

analysis_complete  False  True 
case_month                     
2020-03            76.65  23.35
2020-04            79.20  20.80
2020-05            91.35   8.65
2020-06            93.40   6.60
2020-07            82.92  17.08
2020-08            76.40  23.60
2020-10            91.60   8.40
2020-11            96.80   3.20
